# Cleaning NO₂ Data and Creating a City-Day Dataset (2020–2023)

This notebook cleans raw hourly NO₂ monitoring data from 2020 to 2023 and converts it into a city-day level dataset for analysis and modeling.

### Objectives
- Load raw NO₂ files for 2020–2023
- Clean and standardize column names
- Handle invalid missing values coded as `-999`
- Compute daily station-level NO₂ averages from hourly observations
- Correct mixed date formats across years
- Aggregate station-level records to the city-day level
- Validate the final dataset
- Export the cleaned NO₂ city-day dataset


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import os


##  1: Load Raw NO₂ Files


In [30]:
RAW_DIR = Path("../../data/raw")

file_paths = {
    2020: RAW_DIR / "NO2_2020.csv",
    2021: RAW_DIR / "NO2_2021.csv",
    2022: RAW_DIR / "NO2_2022.csv",
    2023: RAW_DIR / "NO2_2023.csv"
}

no2_list = []

for year, file in file_paths.items():
    try:
        df = pd.read_csv(file, skiprows=7, encoding="latin1")
        no2_list.append(df)
        print(f"Loaded {year}: {file.name} | Shape: {df.shape}")
    except Exception as e:
        print(f"Failed to load {year}: {file}")
        print("Error:", e)

if len(no2_list) == 0:
    raise ValueError("No NO2 files were loaded. Check file paths.")

no2_raw = pd.concat(no2_list, ignore_index=True)

print("\nCombined NO2 shape:", no2_raw.shape)

no2_raw.head()

Loaded 2020: NO2_2020.csv | Shape: (71736, 31)
Loaded 2021: NO2_2021.csv | Shape: (71905, 31)
Loaded 2022: NO2_2022.csv | Shape: (71540, 31)
Loaded 2023: NO2_2023.csv | Shape: (71540, 31)

Combined NO2 shape: (286721, 31)


,Pollutant//Polluant,NAPS ID//Identifiant SNPA,City//Ville,Province/Territory//Province/Territoire,Latitude//Latitude,Longitude//Longitude,Date//Date,H01//H01,H02//H02,H03//H03,...,H15//H15,H16//H16,H17//H17,H18//H18,H19//H19,H20//H20,H21//H21,H22//H22,H23//H23,H24//H24
0,NO2,10102,St. John's,NL,47.56038,-52.71147,1/1/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
1,NO2,10102,St. John's,NL,47.56038,-52.71147,1/2/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
2,NO2,10102,St. John's,NL,47.56038,-52.71147,1/3/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
3,NO2,10102,St. John's,NL,47.56038,-52.71147,1/4/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
4,NO2,10102,St. John's,NL,47.56038,-52.71147,1/5/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999


## 2: Clean Column Names


In [31]:
no2_raw.columns = [c.split("/")[0].strip() for c in no2_raw.columns]
print(no2_raw.columns.tolist())

['Pollutant', 'NAPS ID', 'City', 'Province', 'Latitude', 'Longitude', 'Date', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24']


## 3: Identify Hourly NO₂ Columns


In [32]:
hour_cols = [c for c in no2_raw.columns if c.startswith("H")]

print("Hourly columns found:", hour_cols)
print("Total hourly columns:", len(hour_cols))

Hourly columns found: ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24']
Total hourly columns: 24


## 4: Replace Invalid Missing Codes


In [33]:
no2_raw[hour_cols] = no2_raw[hour_cols].replace(-999, np.nan)

print("Remaining -999 values:", (no2_raw[hour_cols] == -999).sum().sum())
print("Total missing values after replacement:", no2_raw[hour_cols].isna().sum().sum())

Remaining -999 values: 0
Total missing values after replacement: 343354


## 5: Convert Hourly Columns to Numeric


In [34]:
no2_raw[hour_cols] = no2_raw[hour_cols].apply(pd.to_numeric, errors="coerce")

print(no2_raw[hour_cols].dtypes.head())

H01    float64
H02    float64
H03    float64
H04    float64
H05    float64
dtype: object


##  6: Compute Daily NO₂ Average at the Station Level


In [36]:
no2_raw["NO2_daily"] = no2_raw[hour_cols].mean(axis=1)

print(no2_raw[["City", "Date", "NO2_daily"]].head())

         City      Date  NO2_daily
0  St. John's  1/1/2020        NaN
1  St. John's  1/2/2020        NaN
2  St. John's  1/3/2020        NaN
3  St. John's  1/4/2020        NaN
4  St. John's  1/5/2020        NaN


## 7: Keep Required Columns


In [37]:
final = no2_raw[["City", "Date", "NO2_daily"]].copy()
final = final.rename(columns={"NO2_daily": "NO2_daily"})

print(final.shape)
final.head()

(286721, 3)


,City,Date,NO2_daily
0,St. John's,1/1/2020,NaN
1,St. John's,1/2/2020,NaN
2,St. John's,1/3/2020,NaN
3,St. John's,1/4/2020,NaN
4,St. John's,1/5/2020,NaN


## 8: Remove Rows with Missing Daily NO₂ Values



In [41]:
final = final.dropna(subset=["NO2_daily"])

print("Shape after dropping missing NO2_daily:", final.shape)


Shape after dropping missing NO2_daily: (279486, 3)


## 9: Convert the Date Column Properly


In [42]:
final["Date"] = final["Date"].apply(lambda x: pd.to_datetime(x, errors="coerce"))

print("Invalid dates:", final["Date"].isna().sum())

final = final.dropna(subset=["Date"])

final = final[
    (final["Date"] >= "2020-01-01") &
    (final["Date"] <= "2023-12-31")
].copy()

print("Date range:", final["Date"].min(), "to", final["Date"].max())


print("\nRows by parsed calendar year:")
print(final["Date"].dt.year.value_counts().sort_index())

Invalid dates: 0
Date range: 2020-01-01 00:00:00 to 2023-12-31 00:00:00

Rows by parsed calendar year:
Date
2020    70369
2021    70305
2022    69342
2023    69470
Name: count, dtype: int64


## 10: Aggregate to the City-Day Level


In [43]:
final = (
    final.groupby(["City", "Date"], as_index=False)["NO2_daily"]
    .mean()
)

print("Shape after city-day aggregation:", final.shape)
final.head()


Shape after city-day aggregation: (218411, 3)


,City,Date,NO2_daily
0,Agassiz,2020-01-01,1.500000
1,Agassiz,2020-01-02,4.833333
2,Agassiz,2020-01-03,8.750000
3,Agassiz,2020-01-04,5.250000
4,Agassiz,2020-01-05,9.416667


In [44]:
final["Year"] = final["Date"].dt.year
final["Month"] = final["Date"].dt.month

final.head()


,City,Date,NO2_daily,Year,Month
0,Agassiz,2020-01-01,1.500000,2020,1
1,Agassiz,2020-01-02,4.833333,2020,1
2,Agassiz,2020-01-03,8.750000,2020,1
3,Agassiz,2020-01-04,5.250000,2020,1
4,Agassiz,2020-01-05,9.416667,2020,1


In [46]:
print("Final shape:", final.shape)
print("Unique cities:", final["City"].nunique())
print("Date range:", final["Date"].min(), "to", final["Date"].max())

print("\nRows by year:")
print(final["Year"].value_counts().sort_index())

print("\nMissing values:")
print(final.isna().sum())

print("\nDuplicate City-Date rows:", final.duplicated(subset=["City", "Date"]).sum())

print("\nNO2 summary statistics:")
print(final["NO2_daily"].describe())


Final shape: (218411, 5)
Unique cities: 156
Date range: 2020-01-01 00:00:00 to 2023-12-31 00:00:00

Rows by year:
Year
2020    54817
2021    55208
2022    54055
2023    54331
Name: count, dtype: int64

Missing values:
City         0
Date         0
NO2_daily    0
Year         0
Month        0
dtype: int64

Duplicate City-Date rows: 0

NO2 summary statistics:
count    218411.000000
mean          5.159237
std           4.735899
min           0.000000
25%           1.869565
50%           3.760870
75%           6.958333
max          49.347826
Name: NO2_daily, dtype: float64


In [47]:
output_path = "../../data/validated/NO2_cityday.csv"
final.to_csv(output_path, index=False)

print(f"Saved file: {output_path}")

Saved file: ../../data/validated/NO2_cityday.csv
